<a href="https://colab.research.google.com/github/vyom10445/next-word-prediction/blob/main/experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [140]:
#data collection
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg

# load the dataset
data = gutenberg.raw('shakespeare-hamlet.txt')

#save to a file
with open ('hamlet.txt','w')as file:
  file.write(data)

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [141]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

#load the dataset
with open('hamlet.txt','r') as file:
  text = file.read().lower()

# Tokenize the text
Tokenizer()
tokenizer.fit_on_texts([text])
total_words = len(tokenizer.word_index)+1
total_words

4819

In [142]:
input_sequences=[]
for line in text.split('\n'):
  token_list = tokenizer.texts_to_sequences([line])[0]
  for i in range (1,len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

In [143]:
max_sequence_len = max( [len(x) for x in input_sequences])
max_sequence_len

14

In [144]:
input_sequences = np.array(pad_sequences(
    input_sequences,
    maxlen = max_sequence_len,
    padding='pre'
))
input_sequences

array([[   0,    0,    0, ...,    0,    2,  688],
       [   0,    0,    0, ...,    2,  688,    5],
       [   0,    0,    0, ...,  688,    5,   46],
       ...,
       [   0,    0,    0, ...,    5,   46, 1048],
       [   0,    0,    0, ...,   46, 1048,    5],
       [   0,    0,    0, ..., 1048,    5,  194]], dtype=int32)

In [145]:
input_sequences.shape

(25732, 14)

In [146]:
X , y = input_sequences[:,:-1] , input_sequences[:,-1]

In [147]:
X

array([[   0,    0,    0, ...,    0,    0,    2],
       [   0,    0,    0, ...,    0,    2,  688],
       [   0,    0,    0, ...,    2,  688,    5],
       ...,
       [   0,    0,    0, ...,  688,    5,   46],
       [   0,    0,    0, ...,    5,   46, 1048],
       [   0,    0,    0, ...,   46, 1048,    5]], dtype=int32)

In [148]:
y

array([ 688,    5,   46, ..., 1048,    5,  194], dtype=int32)

In [149]:
y = tf.keras.utils.to_categorical(y , num_classes=total_words)
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [150]:
y.shape

(25732, 4819)

In [151]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2)

In [152]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , LSTM , Input , Dropout, Embedding , Bidirectional , GRU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau , ModelCheckpoint
from tensorflow.keras.regularizers import l2

model = Sequential([
    Input(shape=(max_sequence_len-1,)),
    Embedding(total_words, 128),


    Bidirectional(GRU(96, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)),
    Dropout(0.3),

    Bidirectional(GRU(64, dropout=0.2, recurrent_dropout=0.2)),
    Dropout(0.3),


    Dense(256, activation='relu', kernel_regularizer=l2(0.01)),
    Dropout(0.4),

    Dense(total_words, activation='softmax')
])

In [153]:
model.compile(loss='categorical_crossentropy' , optimizer = 'adam' , metrics = ['accuracy'])
model.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_10 (Embedding)        │ (None, 13, 128)        │       616,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 13, 192)        │       130,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 13, 192)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128)            │        99,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 4819)           │     1,238,483 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,117,587 (8.08 MB)

 Trainable params: 2,117,587 (8.08 MB)

 Non-trainable params: 0 (0.00 B)

In [154]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=10,  # Reduced from 15
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,  # Reduced from 5
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_hamlet_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

In [155]:
#train
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - accuracy: 0.0576 - loss: 7.2155
Epoch 1: val_loss improved from None to 6.26876, saving model to best_hamlet_model.h5



Epoch 1: finished saving model to best_hamlet_model.h5
644/644 ━━━━━━━━━━━━━━━━━━━━ 127s 178ms/step - accuracy: 0.0642 - loss: 6.6545 - val_accuracy: 0.0626 - val_loss: 6.2688 - learning_rate: 0.0010
Epoch 2/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step - accuracy: 0.0675 - loss: 6.1775
Epoch 2: val_loss improved from 6.26876 to 6.23482, saving model to best_hamlet_model.h5



Epoch 2: finished saving model to best_hamlet_model.h5
644/644 ━━━━━━━━━━━━━━━━━━━━ 111s 172ms/step - accuracy: 0.0679 - loss: 6.1600 - val_accuracy: 0.0672 - val_loss: 6.2348 - learning_rate: 0.0010
Epoch 3/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.0684 - loss: 6.0328
Epoch 3: val_loss did not improve from 6.23482
644/644 ━━━━━━━━━━━━━━━━━━━━ 141s 171ms/step - accuracy: 0.0708 - loss: 6.0397 - val_accuracy: 0.0666 - val_loss: 6.2490 - learning_rate: 0.0010
Epoch 4/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.0718 - loss: 5.9720
Epoch 4: val_loss did not improve from 6.23482
644/644 ━━━━━━━━━━━━━━━━━━━━ 142s 171ms/step - accuracy: 0.0741 - loss: 5.9584 - val_accuracy: 0.0680 - val_loss: 6.2456 - learning_rate: 0.0010
Epoch 5/50
644/644 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - accuracy: 0.0757 - loss: 5.8783
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 5: val_loss did not improve from 6.23482
644/644 ━━━━━━━━━━━━━━━━━━━